In [88]:
from scipy.special import softmax
import torch
import numpy as np
import torch.nn as nn
from transformers import AutoConfig, AutoModelForSequenceClassification
from safetensors.torch import load_file  # comes with HF if safetensors installed
import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import os
from captum.attr import IntegratedGradients
import torch

In [89]:
ckpt_path = "projects/nace_classification/nace_report_topic_analysis/results/BERT_models/sentiment_test__2__num_layers_1__cos_thres_0.35bert-base-uncased__train_classifier_only__all_labels/checkpoint-1800"
ckpt_path = "projects/nace_classification/nace_report_topic_analysis/results/BERT_models/relevancy_judge__2__num_layers_1__cos_thres_0.35bert-base-uncased__train_classifier_only__all_labels/checkpoint-1408"

In [90]:
# load checkpoint
config = AutoConfig.from_pretrained(ckpt_path)

# 2. Build a model from config (bare BertForSequenceClassification)
model = AutoModelForSequenceClassification.from_config(config)

# 3. Rebuild the SAME classifier architecture as in training
hidden = getattr(config, "custom_hidden", 512)  # fallback if not in config
num_layers = getattr(config, "custom_num_layers", 1)

layers = []
for i in range(num_layers):
    in_dim = config.hidden_size if i == 0 else hidden
    layers.append(nn.Linear(in_dim, hidden))
    layers.append(nn.GELU())
    layers.append(nn.Dropout(0.2))
layers.append(nn.Linear(hidden, 1))
model.classifier = nn.Sequential(*layers)

# 4. Load weights from model.safetensors
state_dict = load_file(os.path.join(ckpt_path, "model.safetensors"))
model.load_state_dict(state_dict, strict=True)  # will fail loudly if mismatch

# 5. Inference mode
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [91]:
tokenizer = AutoTokenizer.from_pretrained(ckpt_path) 

In [92]:
chunk = "registered debt securities debentures and loans as well as other loans are carried at acquisition cost taking into account amortisation or at the lower fair value."
chunk = "stakeholder group from whom complaint is received grievance redressal mechanism in place yes no if yes then provide weblink for grievance redress policy fy current financial year fy current financial year fy current financial year fy previous financial year fy previous financial year fy previous financial year"
chunk = "I love this!"

In [93]:
df_test = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/training_data/test_dataset_sentiment/test_data.csv")
df_test = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/training_data/company_description/test_data.csv")

In [94]:
inputs = tokenizer(chunk, return_tensors="pt", truncation=True)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)
    #logits = softmax(logits)

    probs = torch.sigmoid(logits)   # values between 0 and 1

    # label_scores = {
    #     model.config.id2label[i]: logits[i].item()
    #     for i in range(len(logits))
    # }
    # label_scores = dict(sorted(label_scores.items(), key=lambda x: x[1], reverse=True))
probs

tensor([0.5410])

In [127]:
def binary_class(chunk): 
    inputs = tokenizer(chunk, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)
        #logits = softmax(logits)

        probs = torch.sigmoid(logits)   # values between 0 and 1

        # label_scores = {
        #     model.config.id2label[i]: logits[i].item()
        #     for i in range(len(logits))
        # }
        # label_scores = dict(sorted(label_scores.items(), key=lambda x: x[1], reverse=True))
        # 
    return probs.item() > 0.5

In [128]:
binary_class("HI")

False

In [ ]:
inputs = tokenizer(chunk, return_tensors="pt", truncation=True)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)
    #logits = softmax(logits)

    probs = torch.sigmoid(logits)   # values between 0 and 1

    # label_scores = {
    #     model.config.id2label[i]: logits[i].item()
    #     for i in range(len(logits))
    # }
    # label_scores = dict(sorted(label_scores.items(), key=lambda x: x[1], reverse=True))
probs

In [95]:
logits

tensor([0.1642])

In [103]:
probs_list = []
probs_list_only = []
for i, row in df_test.iterrows():
    inputs = tokenizer(row["text"], return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)
        #logits = softmax(logits)
        probs = torch.sigmoid(logits)   # values between 0 and 1
        probs_list.append([row["label"], float(probs.float())])
        probs_list_only.append(float(probs.float()))

    print(row["label"], probs)

False tensor([0.4537])
False tensor([0.3016])
False tensor([0.3836])
False tensor([0.4820])
False tensor([0.3436])
True tensor([0.6395])
False tensor([0.4447])
True tensor([0.7618])
True tensor([0.5839])
False tensor([0.3334])
False tensor([0.4338])
False tensor([0.4362])
False tensor([0.6488])
True tensor([0.3473])
True tensor([0.5695])
False tensor([0.3140])
False tensor([0.3607])
True tensor([0.5791])
False tensor([0.3754])
False tensor([0.3728])
False tensor([0.4891])
True tensor([0.7944])
False tensor([0.2699])
False tensor([0.1592])
False tensor([0.1596])
False tensor([0.4944])
True tensor([0.4677])
True tensor([0.4540])
False tensor([0.3361])
True tensor([0.7410])
True tensor([0.6211])
False tensor([0.7228])
True tensor([0.7581])
True tensor([0.4023])
True tensor([0.2580])
False tensor([0.2074])
False tensor([0.4068])
True tensor([0.5905])
False tensor([0.4175])
False tensor([0.5492])
False tensor([0.5189])
True tensor([0.7057])
False tensor([0.3172])
True tensor([0.7559])
True 

In [121]:
df

,0,1,predicted,model_right
0,False,0.453709,False,True
1,False,0.301579,False,True
2,False,0.383646,False,True
3,False,0.482016,False,True
4,False,0.343565,False,True
...,...,...,...,...
227,True,0.599637,True,True
228,False,0.345599,False,True
229,True,0.734920,True,True
230,False,0.602378,True,False


In [120]:
probs_list_only

[0.4537089765071869,
 0.3015790581703186,
 0.38364577293395996,
 0.4820159375667572,
 0.34356483817100525,
 0.6394908428192139,
 0.44466671347618103,
 0.7618184685707092,
 0.5839362144470215,
 0.3333525061607361,
 0.43381810188293457,
 0.43617725372314453,
 0.6487712860107422,
 0.34728264808654785,
 0.5694951415061951,
 0.31396031379699707,
 0.3606586754322052,
 0.5791346430778503,
 0.3754354417324066,
 0.3727993965148926,
 0.4890642464160919,
 0.7944398522377014,
 0.2699219882488251,
 0.15924961864948273,
 0.15964509546756744,
 0.49443385004997253,
 0.4677141308784485,
 0.4539719521999359,
 0.33613258600234985,
 0.7409549951553345,
 0.6210764050483704,
 0.7227963805198669,
 0.7581321001052856,
 0.40230292081832886,
 0.25795483589172363,
 0.20742718875408173,
 0.4067749083042145,
 0.5905264019966125,
 0.4175485074520111,
 0.5492413640022278,
 0.5189124345779419,
 0.7057479619979858,
 0.31724655628204346,
 0.7559267282485962,
 0.6428773403167725,
 0.5710910558700562,
 0.2265459001064300

In [119]:
np.array(probs_list_only) > 0.5

array([False, False, False, False, False,  True, False,  True,  True,
       False, False, False,  True, False,  True, False, False,  True,
       False, False, False,  True, False, False, False, False, False,
       False, False,  True,  True,  True,  True, False, False, False,
       False,  True, False,  True,  True,  True, False,  True,  True,
        True, False, False, False,  True, False, False,  True, False,
        True,  True,  True,  True,  True,  True,  True,  True, False,
       False, False, False, False,  True,  True, False,  True,  True,
       False,  True, False, False, False,  True, False,  True, False,
       False,  True,  True, False, False,  True, False,  True, False,
        True, False,  True,  True,  True, False,  True,  True,  True,
        True, False, False, False,  True,  True, False, False,  True,
       False,  True, False,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True, False, False, False,  True, False,
       False, False,

In [113]:
df = pd.DataFrame(probs_list)
df["predicted"] = df[1] > 0.5
df["model_right"] = df["predicted"] == df[0]

In [99]:
predictions = model(tokenized_datasets["test"])
predicted_labels = predictions.predictions.argmax(axis=-1)

NameError: name 'tokenized_datasets' is not defined

### Now with captum

In [ ]:
predicted_class = list(label_scores.items())[0][0]
predicted_class_id = int(np.argmax(logits))
predicted_class_id

14

In [ ]:
np.array(inputs["input_ids"])

array([[  101,  5068,  7016, 12012,  2139, 10609, 22662,  1998, 10940,
         2004,  2092,  2004,  2060, 10940,  2024,  3344,  2012,  7654,
         3465,  2635,  2046,  4070, 16095,  7315,  3370,  2030,  2012,
         1996,  2896,  4189,  3643,  1012,   102]])

In [ ]:
inputs = tokenizer(chunk, return_tensors="pt")
input_ids = inputs["input_ids"]          # LONG!
attention_mask = inputs["attention_mask"]

def forward_func(input_ids, attention_mask):
    return model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).logits

# predicted class
pred = forward_func(input_ids, attention_mask).argmax(dim=1).item()

ig = IntegratedGradients(forward_func)
attr = ig.attribute(
    input_ids,
    target=pred,
    additional_forward_args=(attention_mask,)
)

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [ ]:

model.eval()
ig = IntegratedGradients(model)

inputs = tokenizer(chunk, return_tensors="pt")
outputs = model(**inputs)

attributions = ig.attribute(
    
    inputs["input_ids"].long(),
    target=predicted_class_id,
    additional_forward_args=(inputs["attention_mask"],)
)

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [ ]:
type(inputs["input_ids"]), type(predicted_class_id), type(inputs["attention_mask"])

(torch.Tensor, int)